In [0]:
%sql
USE flights_bronze;

### Załadowanie danych źródłowych

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.types import *

flights_schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day", IntegerType(), True),
    StructField("day_of_week", IntegerType(), True),
    StructField("airline", StringType(), True),
    StructField("flight_number", IntegerType(), True),
    StructField("tail_number", StringType(), True),
    StructField("origin_airport", StringType(), True),
    StructField("destination_airport", StringType(), True),
    StructField("scheduled_departure", StringType(), True),
    StructField("departure_time", StringType(), True),
    StructField("departure_delay", IntegerType(), True),
    StructField("taxi_out", IntegerType(), True),
    StructField("wheels_off", StringType(), True),
    StructField("scheduled_time", IntegerType(), True),
    StructField("elapsed_time", IntegerType(), True),
    StructField("air_time", IntegerType(), True),
    StructField("distance", IntegerType(), True),
    StructField("wheels_on", StringType(), True),
    StructField("taxi_in", IntegerType(), True),
    StructField("scheduled_arrival", StringType(), True),
    StructField("arrival_time", StringType(), True),
    StructField("arrival_delay", IntegerType(), True),
    StructField("diverted", IntegerType(), True),
    StructField("cancelled", IntegerType(), True),
    StructField("cancellation_reason", StringType(), True),
    StructField("air_system_delay", IntegerType(), True),
    StructField("security_delay", IntegerType(), True),
    StructField("airline_delay", IntegerType(), True),
    StructField("late_aircraft_delay", IntegerType(), True),StructField("weather_delay", IntegerType(), True)
])

flights_bronze = "flights_bronze.flights"

df_flights = spark.read.format("csv") \
    .schema(flights_schema) \
    .option("header", "true") \
    .load("/Volumes/workspace/hurtownie_danych/cw7-projekt/flights.csv")

df_flights.write.mode("overwrite").saveAsTable(flights_bronze)

In [0]:
%python
# show a few rows of df_airlines
display(df_flights.limit(5))

year,month,day,day_of_week,airline,flight_number,tail_number,origin_airport,destination_airport,scheduled_departure,departure_time,departure_delay,taxi_out,wheels_off,scheduled_time,elapsed_time,air_time,distance,wheels_on,taxi_in,scheduled_arrival,arrival_time,arrival_delay,diverted,cancelled,cancellation_reason,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay
2015,1,1,4,AS,98,N407AS,ANC,SEA,0005,2354,-11,21,0015,205,194,169,1448,0404,4,0430,0408,-22,0,0,null,null,null,null,null,null
2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,0010,0002,-8,12,0014,280,279,263,2330,0737,4,0750,0741,-9,0,0,null,null,null,null,null,null
2015,1,1,4,US,840,N171US,SFO,CLT,0020,0018,-2,16,0034,286,293,266,2296,0800,11,0806,0811,5,0,0,null,null,null,null,null,null
2015,1,1,4,AA,258,N3HYAA,LAX,MIA,0020,0015,-5,15,0030,285,281,258,2342,0748,8,0805,0756,-9,0,0,null,null,null,null,null,null
2015,1,1,4,AS,135,N527AS,SEA,ANC,0025,0024,-1,11,0035,235,215,199,1448,0254,5,0320,0259,-21,0,0,null,null,null,null,null,null


### Proces ładowania przyrostowego

**Logika *APPEND* - dodanie nowych wierszy.**  
Tabela *flights* zawiera nasze fakty - informacje o lotach które się odbyły (lub zostały przekierowane/odwołane). Jeśli tutaj dodamy jakeiś rekordy, to będą to tylko i wyłącznie informacje o NOWYCH lotach, a nie aktualizacja danych historycznych.


In [0]:
%python
flights_bronze = "flights_bronze.flights"
new_file_path = "/Volumes/workspace/hurtownie_danych/cw7-projekt/flights_new.csv"

# sprawdzamy czy nowy plik istnieje
file_exists = False
try:
  dbutils.fs.ls(new_file_path) # rzuci wyjątek, jeśli nie istnieje
  file_exists = True
except Exception as e:
  if "No such file or directory" in str(e):
    pass
  else:
    raise e

if file_exists:
    df_flights_new = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(new_file_path)

    if not spark.catalog.tableExists(flights_bronze):
        df_flights_new.write.format("delta").saveAsTable(flights_bronze)
    else:
        df_flights_new.write.mode("append").insertInto(flights_bronze) # tu po prostu trzeba dopisać wiersze
else:
  print(f"File {new_file_path} does not exist or have been already processed.")

File /Volumes/workspace/hurtownie_danych/cw7-projekt/flights_new.csv does not exist or have been already processed.
